In [21]:
import cv2
import math
import mediapipe as mp
import matplotlib.pyplot as plt

#### Image

In [11]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='pose_landmarker_heavy.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_poses=1
)

INDICES = {
    "cotovelo_esq": 13,   "cotovelo_dir": 14,
    "mao_esq": 15,        "mao_dir": 16,    
}

CONEXOES = [
    ("cotovelo_esq", "mao_esq"),
    ("cotovelo_dir", "mao_dir"),
]

CALC = [
    ("cotovelo_esq", "mao_esq"),
    ("cotovelo_dir", "mao_dir")
]

In [12]:
def get_ang(caminho_imagem):
    cv2_image = cv2.imread(caminho_imagem)
    altura, largura, _ = cv2_image.shape
    rgb_image = cv2.cvtColor(cv2_image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    with PoseLandmarker.create_from_options(options) as landmarker:
        resultado = landmarker.detect(mp_image)
        
        if resultado.pose_landmarks:
            landmarks = resultado.pose_landmarks[0]
    
            coords = {}
            
            for nome_ponto, indice in INDICES.items():
                lm = landmarks[indice]
                
                x_pixel = int(lm.x * largura)
                y_pixel = int(lm.y * altura)
    
                coords[nome_ponto] = (x_pixel, y_pixel)
                
                cv2.circle(cv2_image, (x_pixel, y_pixel), 2, (0, 255, 0), -1)
    
            for a, b in CALC:
                x_a, y_a = coords[a]
                x_b, y_b = coords[b]
    
                d_x = abs(x_b - x_a)
                d_y = abs(y_b - y_a)
    
                ang_rad = math.atan(d_x / d_y)
                ang = math.degrees(ang_rad)
    
                rounded = round(ang, 2)
    
                cv2.line(cv2_image, (x_a, y_a), (x_b, y_b), (0, 255, 0), 2)
    
                cv2.line(cv2_image, (x_b, y_b), (x_b, y_a), (0, 255, 0), 2)
    
                cv2.line(cv2_image, (x_a, y_a), (x_b, y_a), (0, 255, 0), 2)
    
                if a == 'cotovelo_esq':
                    cv2.putText(cv2_image, f"{rounded}°", (x_b + 10, y_a), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
                else:
                    cv2.putText(cv2_image, f"{rounded}°", (x_b - 10, y_a), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

    cv2.imwrite(f"{caminho_imagem}_result.png", cv2_image)


In [13]:
get_ang('t1.png')
get_ang('t2.png')
get_ang('t3.png')

#### Video

In [57]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='pose_landmarker_heavy.task'),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1
)

INDICES = {
    "cotovelo_esq": 13,   "cotovelo_dir": 14,
    "mao_esq": 15,        "mao_dir": 16,  
    "quadril_esq": 23,    "quadril_dir": 24
}

In [73]:
def encontrar_frames_arremesso(caminho_video, braco_arremesso="mao_dir"):
    cap = cv2.VideoCapture(caminho_video)
    fps = cap.get(cv2.CAP_PROP_FPS)
    altura_video = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    
    historico = []
    frame_atual = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            timestamp_ms = int((frame_atual / fps) * 1000)
            
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
            
            # Usar detect_for_video em vez de detect
            resultado = landmarker.detect_for_video(mp_image, timestamp_ms)
            
            if resultado.pose_landmarks:
                landmarks = resultado.pose_landmarks[0]
                
                # Pegamos apenas a coordenada da mão que faz o arremesso
                lm_mao = landmarks[INDICES[braco_arremesso]]
                punho_y  = int(lm_mao.y * frame.shape[0])

                lm_quadril_e = landmarks[INDICES["quadril_esq"]]
                lm_quadril_d = landmarks[INDICES["quadril_dir"]]
                
                cintura_y = int(((lm_quadril_e.y + lm_quadril_d.y) / 2) * altura_video)
                
                historico.append({
                    "frame": frame_atual,
                    "punho_y": punho_y,
                    "cintura_y": cintura_y
                })
                
            frame_atual += 1

    cap.release()

    idx_soltura = min(range(len(historico)), key=lambda i: historico[i]['punho_y'])
    frame_soltura = historico[idx_soltura]['frame']
    
    frame_preparo = frame_soltura
    maior_y_cintura = 0

    for i in range(idx_soltura, -1, -1):
        cintura_atual = historico[i]['cintura_y']
        
        if cintura_atual > maior_y_cintura:
            maior_y_cintura = cintura_atual
            frame_preparo = historico[i]['frame']

    return frame_preparo, frame_soltura

In [74]:
def calc_ang(caminho_video, frame_preparo):

    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode
    
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path='pose_landmarker_heavy.task'),
        running_mode=VisionRunningMode.IMAGE,
        num_poses=1
    )
    
    cap = cv2.VideoCapture(caminho_video)

    # 1. Pular direto para o frame de preparo e extrair a imagem
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_preparo)
    ret_preparo, img_preparo = cap.read()

    cap.release()

    altura, largura, _ = img_preparo.shape
    rgb_image = cv2.cvtColor(img_preparo, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    with PoseLandmarker.create_from_options(options) as landmarker:
        resultado = landmarker.detect(mp_image)
        
        if resultado.pose_landmarks:
            landmarks = resultado.pose_landmarks[0]
    
            coords = {}
            
            for nome_ponto, indice in INDICES.items():
                lm = landmarks[indice]
                
                x_pixel = int(lm.x * largura)
                y_pixel = int(lm.y * altura)
    
                coords[nome_ponto] = (x_pixel, y_pixel)
                
                cv2.circle(img_preparo, (x_pixel, y_pixel), 2, (0, 255, 0), -1)
    
            for a, b in CALC:
                x_a, y_a = coords[a]
                x_b, y_b = coords[b]
    
                d_x = abs(x_b - x_a)
                d_y = abs(y_b - y_a)
    
                ang_rad = math.atan(d_x / d_y)
                ang = math.degrees(ang_rad)
    
                rounded = round(ang, 2)
    
                cv2.line(img_preparo, (x_a, y_a), (x_b, y_b), (0, 255, 0), 2)
    
                cv2.line(img_preparo, (x_b, y_b), (x_b, y_a), (0, 255, 0), 2)
    
                cv2.line(img_preparo, (x_a, y_a), (x_b, y_a), (0, 255, 0), 2)
    
                if a == 'cotovelo_esq':
                    cv2.putText(img_preparo, f"{rounded}°", (x_b + 10, y_a), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
                else:
                    cv2.putText(img_preparo, f"{rounded}°", (x_b - 10, y_a), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

    cv2.imwrite(f"{caminho_video}_result.png", img_preparo)
    

In [75]:
frame_preparo, frame_soltura = encontrar_frames_arremesso('test2.mp4')
calc_ang('test2.mp4', frame_preparo)